# Classical DE + Enrichment Baseline (SEA_AD)

This notebook runs the classical baseline on **SEA_AD** using the same logic as `de_enrichment.py`:

- DE genes from training cells only (`construct_gene_list`)
- Enrichment on GO BP / Reactome / immune terms
- Redundancy collapse (Jaccard) and top-K non-redundant terms
- Per-cell pathway scoring (`scanpy.tl.score_genes`)
- Logistic regression
- Cell-level and donor-level AUC

Outputs are saved to `SEA_AD/enrichment/`.

In [1]:
# If needed:
# !pip install scanpy statsmodels scikit-learn scipy matplotlib seaborn

import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from SDAN.preprocess import construct_gene_list, qc

warnings.filterwarnings("ignore")
np.random.seed(888)
sc.settings.verbosity = 1

/Users/zxlin/Documents/GitHub/SDAN/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------
# Configuration
# ------------------------
ROOT = Path('.').resolve()
DATASET = 'Astro'   # 'Astro' or 'Micro-PVM'
SEED = 888

DE_PADJ_CUTOFF = 0.05
MIN_OVERLAP = 5
JACCARD_THRESHOLD = 0.5
TOP_K = 40
N_TOP_GENES = 1000

data_dir = ROOT / 'SEA_AD'
anno_dir = ROOT / 'Annotation'
out_dir = data_dir / 'enrichment'
out_dir.mkdir(exist_ok=True)

print('ROOT:', ROOT)
print('DATASET:', DATASET)

ROOT: /Users/zxlin/Documents/GitHub/SDAN
DATASET: Astro


In [3]:
# ------------------------
# Helpers
# ------------------------

def parse_gmt(path):
    terms = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            term = parts[0]
            genes = set(g for g in parts[2:] if g)
            if genes:
                terms[term] = genes
    return terms


def jaccard(a, b):
    union = len(a | b)
    if union == 0:
        return 0.0
    return len(a & b) / union


def median_pairwise_jaccard(term_genes):
    vals = []
    n = len(term_genes)
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(jaccard(term_genes[i], term_genes[j]))
    return float(np.median(vals)) if vals else 0.0

In [4]:
# ------------------------
# Load SEA_AD data and construct train/test split
# ------------------------

np.random.seed(SEED)
h5ad_path = data_dir / 'data' / f'{DATASET}.h5ad'
donor_path = data_dir / 'data' / 'sea-ad_cohort_donor_metadata_082222.xlsx'
if not donor_path.exists():
    donor_path = data_dir / 'data' / 'sea-ad_cohort_donor_metadata_020624.xlsx'

adata = sc.read(h5ad_path)
meta_ind = pd.read_excel(donor_path)

# Same preprocessing entry used in SEA_AD.py
qc(adata)

ind_t = meta_ind[meta_ind['Cognitive Status'] == 'Dementia']['Donor ID']
ind_c = meta_ind[meta_ind['Cognitive Status'] == 'No dementia']['Donor ID']

adata.obs['cell_type'] = adata.obs['Cognitive status']
if 'feature_name' in adata.var.columns:
    adata.var_names = adata.var['feature_name'].astype(str)

mito = pd.read_csv(anno_dir / 'mito_genes.tsv', sep='	')
adata = adata[:, ~adata.var_names.isin(mito['hgnc_symbol'])].copy()

nonzero_prop = np.asarray((adata.X != 0).sum(axis=0)).ravel() / adata.shape[0]
adata = adata[:, nonzero_prop > 0.02].copy()

adata = adata[adata.obs['donor_id'].isin(pd.concat([ind_t, ind_c]))].copy()

test_ind = pd.concat([
    ind_t.sample(n=math.floor(0.5 * len(ind_t))),
    ind_c.sample(n=math.floor(0.5 * len(ind_c))),
])
train_ind = pd.concat([ind_t, ind_c]).drop(test_ind.index)

train_adata = adata[adata.obs['donor_id'].isin(train_ind)].copy()
test_adata = adata[adata.obs['donor_id'].isin(test_ind)].copy()

keep = ['No dementia', 'Dementia']
train_adata = train_adata[train_adata.obs['cell_type'].isin(keep)].copy()
test_adata = test_adata[test_adata.obs['cell_type'].isin(keep)].copy()

print('Train cells:', train_adata.n_obs, 'Test cells:', test_adata.n_obs)
print('Train donors:', train_adata.obs['donor_id'].nunique(), 'Test donors:', test_adata.obs['donor_id'].nunique())
print('Train label counts\n', train_adata.obs['cell_type'].value_counts())

Train cells: 34572 Test cells: 32847
Train donors: 42 Test donors: 42
Train label counts
 cell_type
Dementia       19321
No dementia    15251
Name: count, dtype: int64


In [5]:
# ------------------------
# DE on training cells only (construct_gene_list)
# ------------------------

cell_type_list = ['No dementia', 'Dementia']
train_adata.obs['cell_type'] = train_adata.obs['cell_type'].astype('category')
train_adata.obs['cell_type'] = train_adata.obs['cell_type'].cat.set_categories(cell_type_list)

de_genes_idx = construct_gene_list(
    data=train_adata,
    cell_type_list=cell_type_list,
    n_top_genes=N_TOP_GENES,
    method='fdr_bh',
    alpha=DE_PADJ_CUTOFF,
)
de_genes = set(de_genes_idx.astype(str))

de_df = pd.DataFrame({'gene': sorted(de_genes)})
print('DE gene list source: SDAN.preprocess.construct_gene_list')
print('n_top_genes per class (post-FDR):', N_TOP_GENES)
print('Significant+selected DE genes (union):', len(de_genes))
de_df.head()

The number of DE genes for No dementia: 6865
The number of DE genes for Dementia: 2246
The number of DE genes: 2000
DE gene list source: SDAN.preprocess.construct_gene_list
n_top_genes per class (post-FDR): 1000
Significant+selected DE genes (union): 2000


,gene
0,AB015752.3
1,ABAT
2,ABCA1
3,ABCA12
4,ABCA3


In [6]:

# ------------------------
# Enrichment: GO BP + Reactome + immune terms
# ------------------------

gmt_files = {
    "GO_BP": anno_dir / "c5.go.bp.v2023.2.Hs.symbols.gmt",
    "Reactome": anno_dir / "c2.cp.reactome.v2023.2.Hs.symbols.gmt",
    "Immune": anno_dir / "c7.all.v2023.2.Hs.symbols.gmt",
}

terms = {}
for source, path in gmt_files.items():
    d = parse_gmt(path)
    for term, genes in d.items():
        terms[f"{source}::{term}"] = genes

universe = set(train_adata.var_names.astype(str))
N = len(de_genes & universe)
M = len(universe)

records = []
for term_name, genes in terms.items():
    gs = genes & universe
    n = len(gs)
    if n < MIN_OVERLAP:
        continue
    k = len(gs & de_genes)
    if k < MIN_OVERLAP:
        continue
    pval = hypergeom.sf(k - 1, M, n, N)
    records.append((term_name, n, k, pval, gs))

enrich = pd.DataFrame(records, columns=["term", "set_size", "overlap", "pval", "genes"])
if enrich.empty:
    raise RuntimeError("No enriched terms found. Try lowering MIN_OVERLAP or DE thresholds.")

enrich["padj"] = multipletests(enrich["pval"], method="fdr_bh")[1]
enrich = enrich.sort_values(["padj", "pval", "overlap"], ascending=[True, True, False]).reset_index(drop=True)

sig_enrich = enrich[enrich["padj"] < 0.05].copy()
print("Enriched terms (FDR < 0.05):", sig_enrich.shape[0])
sig_enrich.head(10)


Enriched terms (FDR < 0.05): 421


,term,set_size,overlap,pval,genes,padj
0,GO_BP::GOBP_SYNAPTIC_SIGNALING,569,134,7.457748e-14,"{OPHN1, RAC3, SYNE1, FBXO2, MAPK3, SPG11, SLC2...",4.826746e-10
1,GO_BP::GOBP_CELL_MORPHOGENESIS,751,164,1.266530e-13,"{OPHN1, SMO, RAC3, ECT2, NRP1, SYNE1, SPG11, S...",4.826746e-10
2,GO_BP::GOBP_REGULATION_OF_TRANS_SYNAPTIC_SIGNA...,374,97,6.108092e-13,"{OPHN1, SYNE1, FBXO2, MAPK3, CX3CL1, SNCAIP, C...",1.551862e-09
3,GO_BP::GOBP_GENERATION_OF_NEURONS,1114,215,9.274929e-12,"{SMO, OPHN1, RAC3, NRP1, ECT2, PBX2, SYNE1, SP...",1.721976e-08
4,GO_BP::GOBP_NEURON_DEVELOPMENT,885,179,1.129609e-11,"{OPHN1, SMO, RAC3, NRP1, PBX2, SYNE1, SPG11, L...",1.721976e-08
5,GO_BP::GOBP_CELL_ADHESION,945,188,1.482985e-11,"{RAC3, NRP1, NEDD9, RPSA, ATP2C1, ITGA2B, LAMA...",1.883885e-08
6,GO_BP::GOBP_CELL_JUNCTION_ORGANIZATION,582,129,2.065021e-11,"{OPHN1, RAC3, NRP1, ECT2, NEDD9, SPG11, MYADM,...",2.248513e-08
7,GO_BP::GOBP_CIRCULATORY_SYSTEM_PROCESS,362,90,5.030996e-11,"{DDAH1, SLC29A1, ATP8A1, LRP3, HEG1, NOS1AP, S...",4.793281e-08
8,GO_BP::GOBP_NEUROGENESIS,1280,236,9.174461e-11,"{SMO, NRP1, ECT2, PBX2, SEMA7A, IFT88, ARHGAP4...",7.769749e-08
9,GO_BP::GOBP_SYNAPSE_ORGANIZATION,396,95,1.207388e-10,"{OPHN1, RAC3, NRP1, NEDD9, SPG11, CX3CL1, SETD...",9.202712e-08


In [10]:

# ------------------------
# Redundancy collapse (Jaccard) + top-K non-redundant terms
# ------------------------

selected_idx = []
for idx, row in sig_enrich.iterrows():
    g = row["genes"]
    keep = True
    for j in selected_idx:
        g2 = sig_enrich.loc[j, "genes"]
        if jaccard(g, g2) >= JACCARD_THRESHOLD:
            keep = False
            break
    if keep:
        selected_idx.append(idx)
    if len(selected_idx) >= TOP_K:
        break

selected = sig_enrich.loc[selected_idx].copy().reset_index(drop=True)
selected["n_genes_used"] = selected["genes"].apply(len)

selected_gene_sets = selected["genes"].tolist()
median_jacc = median_pairwise_jaccard(selected_gene_sets)

print("Total enriched terms:", sig_enrich.shape[0])
print("Selected non-redundant terms:", selected.shape[0])
print("Median pairwise Jaccard among selected terms:", round(median_jacc, 4))
selected[["term", "set_size", "overlap", "padj", "n_genes_used"]].head(15)


Total enriched terms: 421
Selected non-redundant terms: 40
Median pairwise Jaccard among selected terms: 0.0321


,term,set_size,overlap,padj,n_genes_used
0,GO_BP::GOBP_SYNAPTIC_SIGNALING,569,134,4.826746e-10,569
1,GO_BP::GOBP_CELL_MORPHOGENESIS,751,164,4.826746e-10,751
2,GO_BP::GOBP_GENERATION_OF_NEURONS,1114,215,1.721976e-08,1114
3,GO_BP::GOBP_CELL_ADHESION,945,188,1.883885e-08,945
4,GO_BP::GOBP_CELL_JUNCTION_ORGANIZATION,582,129,2.248513e-08,582
5,GO_BP::GOBP_CIRCULATORY_SYSTEM_PROCESS,362,90,4.793281e-08,362
6,GO_BP::GOBP_NERVOUS_SYSTEM_PROCESS,665,138,3.229149e-07,665
7,GO_BP::GOBP_AXON_DEVELOPMENT,393,91,1.042887e-06,393
8,Immune::GSE37605_TREG_VS_TCONV_NOD_FOXP3_FUSIO...,93,34,1.042887e-06,93
9,GO_BP::GOBP_REGULATION_OF_MONOATOMIC_ION_TRANS...,343,81,2.752889e-06,343


In [52]:
# ------------------------
# Per-cell activity scores + logistic regression
# ------------------------

train_var = set(train_adata.var_names.astype(str))
test_var = set(test_adata.var_names.astype(str))
scored_rows = []
skipped_terms = 0

for _, row in selected.iterrows():
    genes = sorted([g for g in row['genes'] if (g in train_var and g in test_var)])
    if len(genes) == 0:
        skipped_terms += 1
        continue
    score_col = f"term_score_{len(scored_rows):02d}"
    sc.tl.score_genes(train_adata, gene_list=genes, score_name=score_col, random_state=0, use_raw=False)
    sc.tl.score_genes(test_adata, gene_list=genes, score_name=score_col, random_state=0, use_raw=False)

    row_out = row.copy()
    row_out['genes'] = set(genes)
    row_out['score_col'] = score_col
    scored_rows.append(row_out)

if len(scored_rows) == 0:
    raise RuntimeError('No selected enrichment terms had scoreable genes in both train/test data.')

selected = pd.DataFrame(scored_rows).reset_index(drop=True)
if skipped_terms > 0:
    print(f'Skipped {skipped_terms} selected terms with no scoreable genes.')

feature_cols = selected['score_col'].tolist()
X_train = train_adata.obs[feature_cols].to_numpy()
X_test = test_adata.obs[feature_cols].to_numpy()
y_train = (train_adata.obs['cell_type'].values == 'Dementia').astype(int)
y_test = (test_adata.obs['cell_type'].values == 'Dementia').astype(int)

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)

test_cell_prob = clf.predict_proba(X_test)[:, 1]
cell_auc = roc_auc_score(y_test, test_cell_prob)

test_pred_df = pd.DataFrame({
    'donor_id': test_adata.obs['donor_id'].values,
    'label': y_test,
    'pred_prob': test_cell_prob,
})

donor_df = test_pred_df.groupby('donor_id', as_index=False).agg(
    donor_pred_prob=('pred_prob', 'mean'),
    donor_label=('label', 'max'),
    n_cells=('label', 'size'),
)
donor_auc = roc_auc_score(donor_df['donor_label'], donor_df['donor_pred_prob'])

print(f'Cell-level AUC (classical):  {cell_auc:.4f}')
print(f'Donor-level AUC (classical): {donor_auc:.4f}')
donor_df.head()

Cell-level AUC (classical):  0.6232
Donor-level AUC (classical): 0.7075


,donor_id,donor_pred_prob,donor_label,n_cells
0,H19.33.004,0.607901,0,2368
1,H20.33.004,0.528530,1,948
2,H20.33.008,0.529347,0,370
3,H20.33.013,0.525443,0,1064
4,H20.33.017,0.596567,1,954


In [53]:
# ------------------------
# Classical result snapshot
# ------------------------

result = pd.DataFrame([
    {
        'method': 'Classical DE + enrichment baseline',
        'cell_auc': cell_auc,
        'donor_auc': donor_auc,
        'n_selected_terms': selected.shape[0],
        'median_pairwise_jaccard_selected': median_jacc,
        'n_total_enriched_terms': int(sig_enrich.shape[0]),
    }
])
result

,method,cell_auc,donor_auc,n_selected_terms,median_pairwise_jaccard_selected,n_total_enriched_terms
0,Classical DE + enrichment baseline,0.623212,0.707483,40,0.03208,421


In [10]:
# ------------------------
# Save deliverables
# ------------------------

selected['term_genes'] = selected['genes'].apply(lambda gs: ';'.join(sorted(gs)))
selected['term_de_genes'] = selected['genes'].apply(lambda gs: ';'.join(sorted(gs & de_genes)))
selected['n_term_de_genes'] = selected['genes'].apply(lambda gs: len(gs & de_genes))

selected_export = selected[
    ['term', 'set_size', 'overlap', 'pval', 'padj', 'n_genes_used', 'n_term_de_genes', 'score_col', 'term_genes', 'term_de_genes']
].copy()
selected_export.to_csv(out_dir / f'selected_terms_{DATASET}.tsv', sep='	', index=False)

sig_export = sig_enrich[['term', 'set_size', 'overlap', 'pval', 'padj']].copy()
sig_export.to_csv(out_dir / f'all_enriched_terms_{DATASET}.tsv', sep='	', index=False)

donor_df.to_csv(out_dir / f'donor_predictions_{DATASET}.tsv', sep='	', index=False)

de_df.to_csv(out_dir / f'de_genes_{DATASET}.tsv', sep='	', index=False)

summary = pd.Series({
    'cohort': 'sea_ad',
    'dataset': DATASET,
    'cell_auc_classical': cell_auc,
    'donor_auc_classical': donor_auc,
    'n_total_enriched_terms': int(sig_enrich.shape[0]),
    'n_selected_nonredundant_terms': int(selected.shape[0]),
    'median_pairwise_jaccard_selected': float(median_jacc),
})
summary.to_csv(out_dir / f'summary_{DATASET}.tsv', sep='	', header=False)

print('Saved:')
print(out_dir / f'de_genes_{DATASET}.tsv')
print(out_dir / f'selected_terms_{DATASET}.tsv')
print(out_dir / f'all_enriched_terms_{DATASET}.tsv')
print(out_dir / f'donor_predictions_{DATASET}.tsv')
print(out_dir / f'summary_{DATASET}.tsv')
summary

Saved:
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/selected_terms_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/all_enriched_terms_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/comparison_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/donor_predictions_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/summary_cd4_BL.tsv


dataset                               cd4_BL
cell_auc_classical                  0.761921
donor_auc_classical                 0.889881
n_total_enriched_terms                  2072
n_selected_nonredundant_terms             40
median_pairwise_jaccard_selected    0.033307
cell_auc_sdan                       0.896689
donor_auc_sdan                      0.949405
n_modules_sdan                          40.0
dtype: object

## Notes

- This notebook is configured for **SEA_AD**.
- Change `DATASET` to `'Micro-PVM'` to run the other SEA_AD cell type.
- Outputs are written to `SEA_AD/enrichment/`.